# ML-07 — Baseline Action Score and Top-20 Review

This notebook builds a **transparent, rule-based baseline** that a future ML model must beat.

> Skills loaded: `building-baselines` and `flyrank/flyrank-data`.

**Data contract (from ML-04):**
- **Source:** `fact_content_daily_performance`, mid-panel month `2026-03` partition (not the sealed final-month sample).
- **Observation window:** 2026-03-01 → 2026-03-15 (15 days, feature extraction).
- **Outcome window:** 2026-03-16 → 2026-03-31 (16 days, label evaluation).
- **Label:** `is_declining = 1` when `imp_outcome < 0.8 × imp_obs` (impressions decline > 20%).
- **Minimum activity filter:** `imp_obs >= 50` and `gsc_data_available IS TRUE`.

## 1. My rule and its reason codes

**Rule in plain words:**

> A content item is worth reviewing if **(a)** it had meaningful search visibility in the observation window (≥ 50 impressions), **(b)** it shows signs of risk such as very low click-through rate, position outside page 1, or few active days relative to its impression volume, and **(c)** at least one concrete reason code explains WHY.

The score is a transparent weighted sum of four sub-scores — no fitted weights, just a readable formula:

| Component | Weight | Meaning |
|---|---|---|
| `visibility_score` | 0.40 | Percentile rank of log-impressions — prioritises pages that matter |
| `freshness_risk` | 0.25 | Percentile rank of fraction of obs days with zero impressions — stale-signal proxy |
| `position_opportunity` | 0.25 | Nearness to page 1 × visibility — pages close to the top with most to lose |
| `ctr_gap` | 0.10 | Inverse CTR percentile × visibility — low-CTR visible pages |

**Reason codes the rule can output:**

| Code | Fires when |
|---|---|
| `high_volume_declining_activity` | ≥ 200 imp AND active_days ≤ 7 (out of 15) |
| `position_slipping_visible` | avg position 4–20 AND ≥ 100 imp |
| `low_ctr_visible` | CTR < 2% AND ≥ 100 imp |
| `strong_but_fading` | ≥ 500 imp AND active_days ≤ 10 |
| `general_review` | fallback when no other code fires |

In [1]:
# Section 1 — Setup: connect to warehouse data and verify
import os, sys, json, getpass
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve HF_TOKEN securely
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN and sys.platform == 'win32':
    import winreg
    try:
        with winreg.OpenKey(winreg.HKEY_CURRENT_USER, r'Environment') as key:
            HF_TOKEN, _ = winreg.QueryValueEx(key, 'HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

# Download warehouse parquet via huggingface_hub, then query locally with DuckDB
from huggingface_hub import hf_hub_download
parquet_local = hf_hub_download(
    repo_id='FlyRank/internship-warehouse',
    filename='fact_content_daily_performance/month=2026-03/data_0.parquet',
    repo_type='dataset',
    token=HF_TOKEN,
)

con = duckdb.connect()
parquet_sql = parquet_local.replace('\\', '/')
WAREHOUSE_MARCH = "read_parquet('" + parquet_sql + "')"

sanity = con.sql(
    "SELECT COUNT(*) AS total_rows,"
    " MIN(report_date) AS min_date,"
    " MAX(report_date) AS max_date,"
    " COUNT(DISTINCT client_hash_id) AS n_clients,"
    " COUNT(DISTINCT content_hash_id) AS n_content"
    " FROM " + WAREHOUSE_MARCH
).df()
print("Warehouse March 2026 partition:")
print(sanity.to_string(index=False))

Warehouse March 2026 partition:
 total_rows   min_date   max_date  n_clients  n_content
    9841378 2026-03-01 2026-03-31         55     331437


## 2. Build the ranked queue (writes the CSV)

We build the feature frame from the warehouse using the same observation/outcome split from the data contract, compute the transparent baseline score and reason codes, rank everything, and write to `work/outputs/baseline_action_score.csv`.

In [2]:
# Section 2a — Extract feature frame from warehouse
feature_query = (
    "WITH agg AS ("
    " SELECT client_hash_id, content_hash_id,"
    " SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_obs,"
    " SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_obs,"
    " COUNT(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN 1 END) AS active_days_obs,"
    " AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_obs,"
    " SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_outcome"
    " FROM " + WAREHOUSE_MARCH +
    " WHERE gsc_data_available IS TRUE"
    " GROUP BY client_hash_id, content_hash_id"
    " HAVING imp_obs >= 50"
    ") SELECT client_hash_id, content_hash_id,"
    " imp_obs, clk_obs, active_days_obs,"
    " COALESCE(avg_position_obs, 0.0) AS avg_position_obs,"
    " ROUND((clk_obs * 100.0) / imp_obs, 2) AS ctr_obs,"
    " imp_outcome,"
    " CASE WHEN imp_outcome < 0.8 * imp_obs THEN 1 ELSE 0 END AS is_declining"
    " FROM agg"
)

df = con.sql(feature_query).df()
print(f"Feature frame: {len(df):,} content items (imp_obs >= 50, gsc_data_available IS TRUE)")
print(f"\nLabel distribution:")
print(df['is_declining'].value_counts().rename({0: 'Stable/Growth (0)', 1: 'Declining (1)'}))
base_rate = df['is_declining'].mean()
print(f"\nBase rate (overall declining %): {base_rate:.4f} ({base_rate*100:.2f}%)")

Feature frame: 92,548 content items (imp_obs >= 50, gsc_data_available IS TRUE)

Label distribution:
is_declining
Stable/Growth (0)    66039
Declining (1)        26509
Name: count, dtype: int64

Base rate (overall declining %): 0.2864 (28.64%)


In [3]:
# Section 2b — Compute baseline score, reason codes, and rank

def percentile_rank(series):
    values = pd.to_numeric(series, errors='coerce').fillna(0)
    return values.rank(method='average', pct=True).fillna(0)

def normalize(series):
    values = pd.to_numeric(series, errors='coerce').fillna(0)
    mn, mx = values.min(), values.max()
    if mx == mn:
        return pd.Series(np.zeros(len(values)), index=values.index)
    return (values - mn) / (mx - mn)

# Sub-scores (no fitted weights)
df['visibility_score'] = percentile_rank(np.log1p(df['imp_obs']))

obs_days = 15
df['freshness_risk'] = percentile_rank((obs_days - df['active_days_obs']) / obs_days)

has_pos = (df['avg_position_obs'] > 0).astype(int)
df['position_opportunity'] = (
    (1 - normalize(df['avg_position_obs'].clip(lower=1, upper=50)))
    * df['visibility_score']
    * has_pos
)

df['ctr_gap'] = (1 - percentile_rank(df['ctr_obs'])) * df['visibility_score']

# Composite score
df['baseline_score'] = (
    0.40 * df['visibility_score']
    + 0.25 * df['freshness_risk']
    + 0.25 * df['position_opportunity']
    + 0.10 * df['ctr_gap']
).clip(0, 1)

# Reason codes
def assign_reason_codes(row):
    codes = []
    if row['imp_obs'] >= 200 and row['active_days_obs'] <= 7:
        codes.append('high_volume_declining_activity')
    if 4 <= row['avg_position_obs'] <= 20 and row['imp_obs'] >= 100:
        codes.append('position_slipping_visible')
    if row['ctr_obs'] < 2.0 and row['imp_obs'] >= 100:
        codes.append('low_ctr_visible')
    if row['imp_obs'] >= 500 and row['active_days_obs'] <= 10:
        codes.append('strong_but_fading')
    if not codes:
        codes.append('general_review')
    return '|'.join(codes)

df['reason_codes'] = df.apply(assign_reason_codes, axis=1)

# Suggested action
def suggested_action(reason_str):
    reasons = set(reason_str.split('|'))
    if 'high_volume_declining_activity' in reasons:
        return 'urgent_refresh'
    if 'strong_but_fading' in reasons:
        return 'refresh'
    if 'low_ctr_visible' in reasons:
        return 'optimise_ctr'
    if 'position_slipping_visible' in reasons:
        return 'refresh_and_monitor'
    return 'monitor'

df['suggested_action'] = df['reason_codes'].apply(suggested_action)

# Rank
df['baseline_rank'] = df['baseline_score'].rank(method='first', ascending=False).astype(int)
df = df.sort_values('baseline_rank')

print(f"Scored {len(df):,} items. Score range: [{df['baseline_score'].min():.4f}, {df['baseline_score'].max():.4f}]")
print(f"Median score: {df['baseline_score'].median():.4f}")
print(f"\nReason code distribution:")
all_codes = df['reason_codes'].str.split('|').explode()
print(all_codes.value_counts())
print(f"\nAction distribution:")
print(df['suggested_action'].value_counts())

Scored 92,548 items. Score range: [0.0921, 0.9320]
Median score: 0.4445

Reason code distribution:
reason_codes
low_ctr_visible                   76676
position_slipping_visible         45463
general_review                    15254
strong_but_fading                  1561
high_volume_declining_activity     1234
Name: count, dtype: int64

Action distribution:
suggested_action
optimise_ctr           74446
monitor                15254
urgent_refresh          1234
refresh                 1100
refresh_and_monitor      514
Name: count, dtype: int64


In [4]:
# Section 2c — Precision@K evaluation and write CSV

def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

for k in [10, 20, 50, 100]:
    p_at_k = precision_at_k(df['is_declining'].values, df['baseline_score'].values, k)
    print(f"precision@{k:>3d} = {p_at_k:.4f}   (base rate = {base_rate:.4f})")

# Write CSV
output_dir = Path('work/outputs') if Path('work/outputs').exists() else Path(r'c:\Users\ASUS\Desktop\Flyrank internship\work\outputs')
output_dir.mkdir(parents=True, exist_ok=True)
csv_path = output_dir / 'baseline_action_score.csv'

output_cols = [
    'baseline_rank', 'client_hash_id', 'content_hash_id',
    'baseline_score', 'visibility_score', 'freshness_risk',
    'position_opportunity', 'ctr_gap',
    'reason_codes', 'suggested_action',
    'is_declining',
    'imp_obs', 'clk_obs', 'active_days_obs', 'avg_position_obs', 'ctr_obs',
    'imp_outcome',
]
df[output_cols].to_csv(csv_path, index=False)
print(f"\nWrote {len(df):,} rows to {csv_path.name}")

# Save metrics JSON
metrics = {
    'rows': int(len(df)),
    'base_rate': round(float(base_rate), 4),
    'precision_at_10': round(float(precision_at_k(df['is_declining'].values, df['baseline_score'].values, 10)), 4),
    'precision_at_20': round(float(precision_at_k(df['is_declining'].values, df['baseline_score'].values, 20)), 4),
    'precision_at_50': round(float(precision_at_k(df['is_declining'].values, df['baseline_score'].values, 50)), 4),
    'precision_at_100': round(float(precision_at_k(df['is_declining'].values, df['baseline_score'].values, 100)), 4),
    'score_formula': {
        'visibility_score': 0.40,
        'freshness_risk': 0.25,
        'position_opportunity': 0.25,
        'ctr_gap': 0.10,
    },
}
metrics_path = output_dir / 'baseline_metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2, sort_keys=True))
print(f"Wrote metrics to {metrics_path.name}")
print(json.dumps(metrics, indent=2, sort_keys=True))

precision@ 10 = 0.2000   (base rate = 0.2864)
precision@ 20 = 0.2000   (base rate = 0.2864)
precision@ 50 = 0.2000   (base rate = 0.2864)
precision@100 = 0.1700   (base rate = 0.2864)



Wrote 92,548 rows to baseline_action_score.csv
Wrote metrics to baseline_metrics.json
{
  "base_rate": 0.2864,
  "precision_at_10": 0.2,
  "precision_at_100": 0.17,
  "precision_at_20": 0.2,
  "precision_at_50": 0.2,
  "rows": 92548,
  "score_formula": {
    "ctr_gap": 0.1,
    "freshness_risk": 0.25,
    "position_opportunity": 0.25,
    "visibility_score": 0.4
  }
}


## 3. Top-20 review

For each of the top 20 ranked items: the suggested action, reason code, a confidence note, and what would make the pick wrong.

In [5]:
# Section 3 — Top-20 hand review
top20 = df.head(20).copy()

review_rows = []
for _, row in top20.iterrows():
    if row['imp_obs'] >= 500 and row['active_days_obs'] >= 10:
        confidence = 'HIGH — strong baseline volume, many active days'
    elif row['imp_obs'] >= 200:
        confidence = 'MEDIUM — decent volume, directional signal'
    else:
        confidence = 'LOW — marginal volume, noisy signal'

    wrong_reasons = []
    if row['active_days_obs'] <= 3:
        wrong_reasons.append('very few active days may reflect new/seasonal content, not decay')
    if row['avg_position_obs'] == 0:
        wrong_reasons.append('missing position data — possibly no ranking at all')
    if row['imp_obs'] < 100:
        wrong_reasons.append('low volume makes any trend noisy')
    if row['ctr_obs'] > 10:
        wrong_reasons.append('high CTR suggests the page works well — may be a false positive')
    if not wrong_reasons:
        wrong_reasons.append('seasonal/external traffic shift unrelated to content quality')

    review_rows.append({
        'rank': int(row['baseline_rank']),
        'action': row['suggested_action'],
        'reason_codes': row['reason_codes'],
        'imp_obs': int(row['imp_obs']),
        'active_days': int(row['active_days_obs']),
        'ctr_obs': round(row['ctr_obs'], 2),
        'avg_pos': round(row['avg_position_obs'], 1),
        'is_declining': int(row['is_declining']),
        'confidence': confidence,
        'what_would_make_it_wrong': '; '.join(wrong_reasons),
    })

review_df = pd.DataFrame(review_rows)
print("=== TOP-20 HAND REVIEW ===\n")
display_cols = ['rank', 'action', 'reason_codes', 'imp_obs', 'active_days',
                'ctr_obs', 'avg_pos', 'is_declining', 'confidence']
print(review_df[display_cols].to_string(index=False))

top20_precision = top20['is_declining'].mean()
print(f"\nTop-20 actual declining rate: {top20_precision:.4f} ({top20_precision*100:.1f}%)")
print(f"Base rate: {base_rate:.4f} ({base_rate*100:.1f}%)")
if base_rate > 0:
    print(f"Lift over base rate: {top20_precision / base_rate:.2f}x")

=== TOP-20 HAND REVIEW ===

 rank         action                                                                               reason_codes  imp_obs  active_days  ctr_obs  avg_pos  is_declining                                      confidence
    1 urgent_refresh high_volume_declining_activity|position_slipping_visible|low_ctr_visible|strong_but_fading    13576            7     0.00      6.4             0      MEDIUM — decent volume, directional signal
    2        refresh                                                          low_ctr_visible|strong_but_fading     6128           10     0.00      2.0             1 HIGH — strong baseline volume, many active days
    3 urgent_refresh                           high_volume_declining_activity|low_ctr_visible|strong_but_fading     6779            7     0.06      0.5             0      MEDIUM — decent volume, directional signal
    4   optimise_ctr                                                                            low_ctr_visible    3

In [6]:
# Section 3b — Full review with failure modes
print("=== FULL TOP-20 REVIEW (with failure modes) ===\n")
for _, r in review_df.iterrows():
    declining_str = 'YES' if r['is_declining'] else 'no'
    print(f"Rank {int(r['rank']):>4d} | {r['action']:<22s} | {r['reason_codes']}")
    print(f"         imp={int(r['imp_obs']):>6d}  active_days={int(r['active_days']):>2d}  "
          f"ctr={r['ctr_obs']:.2f}%  pos={r['avg_pos']:.1f}  "
          f"actually_declining={declining_str}")
    print(f"         Confidence: {r['confidence']}")
    print(f"         Wrong if: {r['what_would_make_it_wrong']}")
    print()

=== FULL TOP-20 REVIEW (with failure modes) ===

Rank    1 | urgent_refresh         | high_volume_declining_activity|position_slipping_visible|low_ctr_visible|strong_but_fading
         imp= 13576  active_days= 7  ctr=0.00%  pos=6.4  actually_declining=no
         Confidence: MEDIUM — decent volume, directional signal
         Wrong if: seasonal/external traffic shift unrelated to content quality

Rank    2 | refresh                | low_ctr_visible|strong_but_fading
         imp=  6128  active_days=10  ctr=0.00%  pos=2.0  actually_declining=YES
         Confidence: HIGH — strong baseline volume, many active days
         Wrong if: seasonal/external traffic shift unrelated to content quality

Rank    3 | urgent_refresh         | high_volume_declining_activity|low_ctr_visible|strong_but_fading
         imp=  6779  active_days= 7  ctr=0.06%  pos=0.5  actually_declining=no
         Confidence: MEDIUM — decent volume, directional signal
         Wrong if: seasonal/external traffic shift un

## 4. Weak picks + leakage check

### Weak picks

Reviewing the top-20, we expect some picks that look wrong — the skill says "if it found none, look harder." Below we systematically identify weak picks and explain why.

### Leakage check

Confirm no product flags or future/outcome-window data leaked into the score.

In [7]:
# Section 4a — Identify weak picks in the top-20
print("=== WEAK PICKS IN THE TOP-20 ===\n")

weak_count = 0
for _, row in top20.iterrows():
    issues = []
    if row['is_declining'] == 0 and row['imp_obs'] >= 300:
        issues.append('NOT declining despite high volume — the baseline over-ranks visible-but-stable pages')
    if row['active_days_obs'] <= 3 and row['imp_obs'] < 150:
        issues.append('Very few active days + low volume — could be new/seasonal, not decaying')
    if row['avg_position_obs'] == 0:
        issues.append('No position data — position_opportunity sub-score contributes nothing')
    if row['ctr_obs'] > 8 and row['is_declining'] == 0:
        issues.append('High CTR and not declining — this page is performing well, likely a false alarm')
    if issues:
        weak_count += 1
        print(f"Rank {int(row['baseline_rank']):>4d} (declining={int(row['is_declining'])}):")
        for issue in issues:
            print(f"  - {issue}")
        print()

fp_count = int((top20['is_declining'] == 0).sum())
print(f"Found {weak_count} weak pick(s) in the top 20.")
print(f"{fp_count} of top-20 are NOT actually declining (false positives).")
print(f"\nThe baseline's precision@20 ({top20_precision:.2f}) is BELOW the base rate ({base_rate:.2f}).")
print("This is an honest result: the rule prioritises high-impression pages, which tend")
print("to be more stable. A trained model should easily beat this baseline by learning")
print("subtler patterns (activity-day trends, CTR changes, position drift).")

=== WEAK PICKS IN THE TOP-20 ===

Rank    1 (declining=0):
  - NOT declining despite high volume — the baseline over-ranks visible-but-stable pages

Rank    3 (declining=0):
  - NOT declining despite high volume — the baseline over-ranks visible-but-stable pages

Rank    4 (declining=0):
  - NOT declining despite high volume — the baseline over-ranks visible-but-stable pages

Rank    5 (declining=0):
  - NOT declining despite high volume — the baseline over-ranks visible-but-stable pages

Rank    6 (declining=0):
  - NOT declining despite high volume — the baseline over-ranks visible-but-stable pages

Rank    7 (declining=0):
  - NOT declining despite high volume — the baseline over-ranks visible-but-stable pages

Rank    9 (declining=0):
  - NOT declining despite high volume — the baseline over-ranks visible-but-stable pages

Rank   10 (declining=0):
  - NOT declining despite high volume — the baseline over-ranks visible-but-stable pages

Rank   11 (declining=0):
  - NOT declining des

In [8]:
# Section 4b — Leakage check
print("=== LEAKAGE CHECK ===\n")

feature_inputs = ['imp_obs', 'clk_obs', 'active_days_obs', 'avg_position_obs', 'ctr_obs']
print("Features used in baseline score (all observation-window only, 2026-03-01 to 2026-03-15):")
for feat in feature_inputs:
    print(f"  - {feat}")

print("\nOutcome-window columns present in dataframe but NOT used in score:")
print("  - imp_outcome (used ONLY for label derivation, never as a feature)")
print("  - is_declining (the label — used only for evaluation, never in score)")

corr_score_outcome = df['baseline_score'].corr(df['imp_outcome'])
corr_score_label = df['baseline_score'].corr(df['is_declining'])
print(f"\nCorrelation(baseline_score, imp_outcome) = {corr_score_outcome:.4f}")
print(f"Correlation(baseline_score, is_declining) = {corr_score_label:.4f}")

print("\n--- Logical leakage audit ---")
print("baseline_score = 0.40*visibility + 0.25*freshness_risk + 0.25*position_opp + 0.10*ctr_gap")
print("  visibility_score     = percentile_rank(log1p(imp_obs))           -- obs-window ONLY")
print("  freshness_risk       = percentile_rank((15 - active_days_obs)/15) -- obs-window ONLY")
print("  position_opportunity = (1 - norm(avg_position_obs))*vis*has_pos  -- obs-window ONLY")
print("  ctr_gap              = (1 - percentile_rank(ctr_obs))*vis        -- obs-window ONLY")
print("\nVerdict: NO product flags, NO outcome-window data, NO future-window data used in score.")
print("The label (is_declining) depends only on imp_outcome vs imp_obs — no circular feature leakage.")

=== LEAKAGE CHECK ===

Features used in baseline score (all observation-window only, 2026-03-01 to 2026-03-15):
  - imp_obs
  - clk_obs
  - active_days_obs
  - avg_position_obs
  - ctr_obs

Outcome-window columns present in dataframe but NOT used in score:
  - imp_outcome (used ONLY for label derivation, never as a feature)
  - is_declining (the label — used only for evaluation, never in score)

Correlation(baseline_score, imp_outcome) = 0.4274
Correlation(baseline_score, is_declining) = -0.0469

--- Logical leakage audit ---
baseline_score = 0.40*visibility + 0.25*freshness_risk + 0.25*position_opp + 0.10*ctr_gap
  visibility_score     = percentile_rank(log1p(imp_obs))           -- obs-window ONLY
  freshness_risk       = percentile_rank((15 - active_days_obs)/15) -- obs-window ONLY
  position_opportunity = (1 - norm(avg_position_obs))*vis*has_pos  -- obs-window ONLY
  ctr_gap              = (1 - percentile_rank(ctr_obs))*vis        -- obs-window ONLY

Verdict: NO product flags, NO ou

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.